# 2.7 MCP: Sharing Tools with the Model Context Protocol

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Learning objectives

- Explain the problem MCP solves: every app wiring its own tools, again and again, in incompatible ways.
- Name the three roles in MCP (host, client, server) and what a server can expose: tools, resources, and prompts.
- Build a small MCP server with FastMCP, as its own file, separate from the agent.
- Connect a LangChain agent to that server with `MultiServerMCPClient`, load its tools, and let the agent call them.
- Tell the two transports apart (stdio for local servers, streamable HTTP for remote ones) and say when each is used.
- Decide honestly when MCP is worth it and when a plain `@tool` in the same file is the better engineering call.
- State the security rule for MCP servers and why tool descriptions deserve suspicion.

## The problem: every agent rewires the same tools

Look back at how this module built tools. In lesson 2.2 you wrote a function, decorated it with `@tool`, and handed it to one agent in one file. That works, and for a single project it is often the right answer. Now imagine the same tool needed in three places: your LangChain agent, your teammate's LangGraph service, and the desktop AI app your support team uses. Without a standard, you wire it three times, in three frameworks, and maintain it in three places.

The Model Context Protocol (MCP) is the standard that removes the rewiring. A tool is written once, behind a server that speaks MCP. Any client that speaks MCP can discover the tool, read its schema, and call it, whatever framework or language either side uses. The comparison people reach for is USB: one plug shape, many devices. Build the tool once, plug it into any agent.

There is a second benefit that matters in real projects: separation. The MCP server is its own program, usually its own file or its own service, owned by whoever owns the capability (the database team, the payments team, you). The agent team consumes it. That is the same multi-file thinking production code always needs, and this lesson is deliberately written that way: the server and the agent are two files, and they never import each other.

## The three roles

- **Server.** A program that exposes capabilities over MCP: **tools** (functions the model can call), **resources** (readable data, like files or database rows the client can fetch), and **prompts** (reusable prompt templates). This lesson uses tools; the other two exist and are worth knowing by name.
- **Client.** The component inside your application that connects to one server, discovers what it offers, and translates between MCP and your framework. In this lesson the client is `MultiServerMCPClient` from `langchain-mcp-adapters`, which turns MCP tools into LangChain tools your agent already understands.
- **Host.** The application the user actually runs, which holds the client (or several). Your agent script is the host here. One host can connect to many servers at once, which is the whole point: tools from five servers appear in your agent as one tool list.

## File 1: the MCP server

MCP servers in Python are usually written with FastMCP, part of the official `mcp` package. The shape will look familiar: it is the `@tool` idea from lesson 2.2, except the decorator belongs to the server, and the function now lives behind a protocol instead of inside your agent.

Save this as `mcp_server.py`:

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("study-tools")

NOTES = {
    "rag": "Retrieval augmented generation: fetch documents first, then answer from them.",
    "lcel": "LangChain Expression Language: pipe runnables together with the | operator.",
}

@mcp.tool()
def lookup_note(topic: str) -> str:
    """Look up a short study note by topic name."""
    return NOTES.get(topic.lower(), f"No note saved for {topic}.")

@mcp.tool()
def word_count(text: str) -> int:
    """Count the words in a piece of text."""
    return len(text.split())

if __name__ == "__main__":
    mcp.run(transport="stdio")

Install the two packages this lesson adds (the rest are the course's pinned versions):

In [ ]:
!pip install mcp==1.30.0 langchain-mcp-adapters==0.3.2

Notice what is not here: no model, no LangChain, no agent. The server knows nothing about who will call it. The docstrings do the same job they did in lesson 2.2: they become the tool descriptions a model reads when deciding what to call. The `transport="stdio"` line says the server talks over standard input and output: the client starts it as a subprocess and speaks to it through pipes. That is the normal shape for a local server.

You can run the server alone (`python mcp_server.py`) and it will sit waiting silently. That silence is correct; it speaks a JSON protocol, not human text, and it is waiting for a client.

## File 2: the agent that connects to it

Save this as `agent.py`, in the same folder:

In [ ]:
import asyncio
import sys

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI


async def main() -> None:
    client = MultiServerMCPClient(
        {
            "study": {
                "transport": "stdio",
                "command": sys.executable,
                "args": ["mcp_server.py"],
            }
        }
    )

    tools = await client.get_tools()
    print("Loaded tools:", [t.name for t in tools])

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    agent = create_agent(model=model, tools=tools)

    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "What does LCEL mean? Use the notes tool."}]}
    )
    print(result["messages"][-1].content)


if __name__ == "__main__":
    asyncio.run(main())

Run it with `python agent.py`. What happens, step by step:

1. The client starts `mcp_server.py` as a subprocess (that is what `command` and `args` describe) and opens an MCP session over stdio.
2. `get_tools()` asks the server what it offers and converts each MCP tool into a LangChain `StructuredTool`. From here on, they behave like the tools you built in lesson 2.2; the agent cannot tell the difference.
3. The agent decides it needs `lookup_note`, the client sends the call over the protocol, the server runs the real function, and the result comes back as a `ToolMessage`.
4. The model answers from the note.

Two practical details worth knowing. First, everything is async: MCP clients are async by design, so the agent code lives inside `asyncio.run`, the same pattern production services use. Second, results from MCP tools can arrive as a list of content blocks (text, images) rather than a plain string, because MCP is framework-neutral. The adapter passes that through, so when you inspect a `ToolMessage` from an MCP tool you may see `[{'type': 'text', 'text': '...'}]` instead of a bare string. The model handles it fine; only your printing code needs to know.

To use several servers, add more entries to the same dict: a `"files"` server, a `"database"` server, each with its own command or URL. `get_tools()` returns all of their tools in one list.

On the free path from lesson 1.4, swap the model line for `ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)`. Nothing else changes; that is the benefit of the adapter doing the translation.

## Transports: stdio and streamable HTTP

- **stdio** is for local servers: the client launches the server process on the same machine. Development tools, desktop integrations, and personal automation live here. The server config is a command plus arguments, as above.
- **Streamable HTTP** is for remote servers: the server runs as a web service, and the config is a URL:

In [ ]:
client = MultiServerMCPClient(
    {
        "study": {
            "transport": "streamable_http",
            "url": "http://localhost:8000/mcp",
        }
    }
)

A FastMCP server opts into it with `mcp.run(transport="streamable-http")`, which serves on port 8000 by default. Use this shape when the server is shared by a team or runs in production. You may also see an older transport named SSE in tutorials; it is deprecated, so do not build on it.

## When MCP is worth it, and when it is not

Honest guidance, because adding a protocol has a cost:

- **Use a plain `@tool`** when the tool lives and dies inside one project. It is less code, easier to debug, and one less process to run. Most of this course is correctly built that way.
- **Use MCP** when a capability is shared across projects, teams, or frameworks; when the tool is owned by a different codebase or service than the agent; or when you want to consume tools that already exist as MCP servers (file systems, databases, GitHub, and many SaaS products now publish them).
- The mistake to avoid is wrapping your own single-project function in a server out of fashion. A protocol boundary earns its keep when something real crosses it.

## Security: a server is code you are trusting

A connected MCP server is not a passive data source. Its tools run with real effects, and its tool descriptions are text your model reads, which means a hostile or careless server can describe its tools in ways that steer your agent. The rules:

1. Only connect to servers you trust or have read. Running a stranger's stdio server is running their code on your machine.
2. Treat tool descriptions as untrusted input, the same posture Module 6 takes toward retrieved documents in RAG.
3. Keep the human in the loop for destructive tools. The approvals pattern from Module 5 applies to MCP tools exactly as it does to local ones.

## What can go wrong

- **The server file is not found.** The stdio config launches the command from the working directory. Run `agent.py` from the folder that contains `mcp_server.py`, or pass an absolute path in `args`.
- **Server logs pollute the protocol.** Anything the server prints to stdout corrupts the stdio stream. Print debugging to stderr (`print(..., file=sys.stderr)`) or use logging configured for stderr.
- **Forgetting the async shape.** `get_tools()` and `ainvoke` are coroutines; calling them outside an async function raises immediately. If your app is synchronous, run the client setup once at startup inside `asyncio.run` and keep the loaded tools.
- **Version drift between client and server.** MCP is young and moving. Pin both packages (as this lesson does) and re-test after upgrades.
- **Assuming every server is safe.** See the security section. This is the failure with real consequences.

## Review questions

**1. The MCP server in this lesson never imports LangChain. Why is that the point, not a limitation?**
Because the protocol is the contract, not the framework. The same `mcp_server.py` can serve a LangChain agent, a LangGraph service, or a desktop AI app without changes. The moment the server depended on one framework, the shared-tool problem would come back for everyone else.

**2. Your agent calls an MCP tool and the `ToolMessage` content is a list of blocks instead of a string. Is something broken?**
No. MCP is framework-neutral, so results travel as typed content blocks (text, image, and so on). The adapter passes them through, and the model reads them normally. Only code that assumes a plain string, usually your own logging or printing, needs adjusting.

**3. A teammate proposes converting every tool in the Module 2 mini-project into MCP servers before the capstone. What do you say?**
Decline, with the lesson's rule: nothing crosses a boundary inside a single project, so the protocol adds processes and failure modes for no benefit. The capstone's structure requirement is about organising files inside the project, which plain `@tool` modules satisfy. Revisit MCP when a second consumer for the same tool actually appears.

## Exercise

Take the personal task agent from lesson 2.6 and split one capability out: move its task-store reads (`list_tasks` and `search_tasks`) into a small MCP server named `task_server.py`, backed by the same JSON file. Reconnect the agent through `MultiServerMCPClient` alongside its remaining local tools. Acceptance: the agent answers "what is on my list?" using the MCP tool (check the trace), the local tools still work in the same conversation, and you can run the server alone and see it wait silently. Then write two sentences in your notes: what did the split cost you, and when would it have been worth it?

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.